# Proyecto Big Data - Evaluación Sumativa 3
## Caso Empresarial: StreamData Analytics
### Ingesta, Procesamiento ETL y Análisis de Sentimiento del Catálogo Netflix
**Integrantes:** Bastián Martínez & Pablo Amión  
**Institución:** INACAP - Sede Puente Alto  
**Arquitectura:** Amazon S3 (Data Lake) + Amazon Athena (SQL Serverless) + Microsoft Power BI
---
Este notebook implementa el ciclo de vida del dato para la fuente histórica `netflix_titles.csv`, preparando los datos para su almacenamiento en las capas **RAW**, **PROCESSED** y **CURATED** del Data Lake.

### 1. Importación de Librerías y Configuración del Entorno

In [1]:
import os
import json
import datetime
import warnings
import pandas as pd
import numpy as np
from textblob import TextBlob

warnings.filterwarnings('ignore')

# Crear directorios locales para estructurar el Data Lake
os.makedirs('salida_netflix/processed', exist_ok=True)
os.makedirs('salida_netflix/curated', exist_ok=True)
os.makedirs('salida_netflix/logs', exist_ok=True)

print('Entorno configurado correctamente.')

### 2. Extracción y Validación de la Fuente (Zona RAW)
Se carga el dataset original `netflix_titles.csv` descargado desde Kaggle, validando el esquema de 12 columnas y la integridad del identificador `show_id`.

In [2]:
csv_path = 'netflix_titles.csv'
df_raw = pd.read_csv(csv_path, encoding='utf-8')

print(f'Dimensiones originales: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas')
print(f'Identificadores nulos: {df_raw["show_id"].isna().sum()}')
print(f'Identificadores duplicados: {df_raw.duplicated(subset=["show_id"]).sum()}')
df_raw.head(3)

### 3. Limpieza de Datos y Reparaciones Estructurales
Se corrigen tres anomalías estructurales donde la duración quedó desplazada en la columna `rating` (`s5542`, `s5795`, `s5814`), se estandarizan las cadenas y se parsean las fechas a formato ISO 8601 (`YYYY-MM-DD`).

In [3]:
df = df_raw.copy()

# Normalización de espacios
for col in df.columns:
    if df[col].dtype == object or str(df[col].dtype).startswith('str'):
        df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
df.replace({'nan': None, '': None}, inplace=True)

# Reparar anomalía en rating / duration
anomaly_mask = df['rating'].astype(str).str.contains(r'\d+\s*min', na=False) & df['duration'].isna()
print(f'Registros con duración en rating detectados: {anomaly_mask.sum()}')
df.loc[anomaly_mask, 'duration'] = df.loc[anomaly_mask, 'rating']
df.loc[anomaly_mask, 'rating'] = 'No informado'

# Imputación de faltantes categóricos
for c in ['director', 'cast', 'country', 'rating', 'duration']:
    df[c] = df[c].fillna('No informado')

# Parseo de date_added
parsed_dates = pd.to_datetime(df['date_added'].astype(str).str.strip(), format='mixed', errors='coerce')
df['date_added_clean'] = parsed_dates.dt.strftime('%Y-%m-%d')
df['added_year'] = parsed_dates.dt.year.astype('Int64')
df['added_month'] = parsed_dates.dt.month.astype('Int64')

print(f'Fechas nulas válidas conservadas: {parsed_dates.isna().sum()}')

### 4. Enriquecimiento de Atributos y Análisis Léxico con TextBlob
Se clasifican los grupos de audiencia, se calcula la duración numérica y se aplica procesamiento de lenguaje natural (NLP) sobre la columna `description` para determinar la polaridad léxica.

In [4]:
df.rename(columns={'type': 'content_type'}, inplace=True)

# Atributos derivados
df['primary_country'] = df['country'].apply(lambda x: 'No informado' if x == 'No informado' else x.split(',')[0].strip())
df['country_count'] = df['country'].apply(lambda x: 0 if x == 'No informado' else len([c for c in x.split(',') if c.strip()]))
df['primary_genre'] = df['listed_in'].apply(lambda x: 'No informado' if pd.isna(x) else x.split(',')[0].strip())
df['genre_count'] = df['listed_in'].apply(lambda x: 0 if pd.isna(x) else len([g for g in str(x).split(',') if g.strip()]))
df['release_decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Grupo de audiencia
def map_audience(rating):
    r = str(rating).upper().strip()
    if r in ['TV-MA', 'R', 'NC-17', 'UR']: return 'Adultos'
    elif r in ['PG-13', 'TV-14']: return 'Adolescentes'
    elif r in ['PG', 'TV-PG']: return 'Supervisión parental'
    elif r in ['TV-Y7', 'TV-Y7-FV']: return 'Infantil'
    elif r in ['G', 'TV-G', 'TV-Y']: return 'General'
    else: return 'Sin clasificación'
df['audience_group'] = df['rating'].apply(map_audience)

# Duración numérica y unidad
def parse_duration(d):
    if not d or d == 'No informado': return 0, 'No informado'
    parts = str(d).split()
    val = int(parts[0]) if parts[0].isdigit() else 0
    unit = 'min' if 'min' in str(d).lower() else 'season'
    return val, unit
dur_res = df['duration'].apply(parse_duration)
df['duration_value'] = [p[0] for p in dur_res]
df['duration_unit'] = [p[1] for p in dur_res]

# Polaridad léxica con TextBlob
def analyze_sentiment(text):
    if not text or text == 'No informado': return 0.0, 'Neutro', 0
    blob = TextBlob(str(text))
    pol = blob.sentiment.polarity
    wc = len(str(text).split())
    if pol > 0.10: label = 'Positivo'
    elif pol < -0.10: label = 'Negativo'
    else: label = 'Neutro'
    return round(pol, 4), label, wc

sent_res = df['description'].apply(analyze_sentiment)
df['sentiment_polarity'] = [r[0] for r in sent_res]
df['sentiment_label'] = [r[1] for r in sent_res]
df['description_word_count'] = [r[2] for r in sent_res]

print('Distribución de tono de sinopsis:')
print(df['sentiment_label'].value_counts())

### 5. Generación de Tablas Puente Normalizadas (1:N)
Se resuelven las relaciones multivaluadas de país y género generando dos tablas puente normalizadas que conectan mediante `show_id`.

In [5]:
# Tabla puente países
country_rows = []
for _, row in df[df['country'] != 'No informado'].iterrows():
    for c in str(row['country']).split(','):
        if c.strip(): country_rows.append({'show_id': row['show_id'], 'country_name': c.strip()})
df_countries = pd.DataFrame(country_rows)

# Tabla puente géneros
genre_rows = []
for _, row in df.iterrows():
    for g in str(row['listed_in']).split(','):
        if g.strip(): genre_rows.append({'show_id': row['show_id'], 'genre': g.strip()})
df_genres = pd.DataFrame(genre_rows)

print(f'Tabla netflix_title_countries: {len(df_countries)} filas, {df_countries["country_name"].nunique()} países')
print(f'Tabla netflix_title_genres: {len(df_genres)} filas, {df_genres["genre"].nunique()} géneros')

### 6. Exportación de Tablas para Amazon S3 (Capa Processed y Curated)
Se guardan los CSV estructurados listos para ser consultados por Amazon Athena y consumidos en Power BI.

In [6]:
cols_clean = [
    'show_id', 'content_type', 'title', 'director', 'cast',
    'country', 'primary_country', 'country_count', 'date_added_clean',
    'added_year', 'added_month', 'release_year', 'release_decade',
    'rating', 'audience_group', 'duration', 'duration_value',
    'duration_unit', 'listed_in', 'primary_genre', 'genre_count',
    'description', 'sentiment_polarity', 'sentiment_label', 'description_word_count'
]
df_clean = df[cols_clean].rename(columns={'date_added_clean': 'date_added', 'cast': 'cast_members'})

# Exportar Processed
df_clean.to_csv('salida_netflix/processed/netflix_titles_clean.csv', index=False, encoding='utf-8')
df_countries.to_csv('salida_netflix/processed/netflix_title_countries.csv', index=False, encoding='utf-8')
df_genres.to_csv('salida_netflix/processed/netflix_title_genres.csv', index=False, encoding='utf-8')

# Resumen de sentimiento y KPIs para Curated
sent_vc = df['sentiment_label'].value_counts()
df_sent_summary = pd.DataFrame([
    {'sentiment_label': 'Positivo', 'title_count': int(sent_vc.get('Positivo', 0)), 'percentage': round(sent_vc.get('Positivo', 0)/len(df)*100, 2)},
    {'sentiment_label': 'Neutro', 'title_count': int(sent_vc.get('Neutro', 0)), 'percentage': round(sent_vc.get('Neutro', 0)/len(df)*100, 2)},
    {'sentiment_label': 'Negativo', 'title_count': int(sent_vc.get('Negativo', 0)), 'percentage': round(sent_vc.get('Negativo', 0)/len(df)*100, 2)},
])
df_sent_summary.to_csv('salida_netflix/curated/netflix_sentiment_summary.csv', index=False, encoding='utf-8')

print('Archivos exportados exitosamente en salida_netflix/processed y salida_netflix/curated.')